In [21]:
!pip install groq gradio pdfplumber reportlab streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 103.2 MB/s eta 0:00:00


In [22]:
import os
os.environ["GROQ_API_KEY"] = "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"

In [23]:
import os
import pdfplumber
import gradio as gr
from groq import Groq
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4

# Initialize Groq
client = Groq(api_key=os.environ["GROQ_API_KEY"])


# -------- Extract Resume Text --------
def extract_text_from_pdf(file_path):
    text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text


# -------- Generate ATS Analysis --------
def generate_analysis(resume_text, job_description):

    prompt = f"""
You are an ATS Resume Analyzer.

Return strictly:

ATS Score: <number>/100
Keyword Match Percentage: <number>%
Missing Keywords: <comma separated>

Improvements:
- point 1
- point 2
- point 3

Rewritten Professional Summary:
<improved summary>

Job Description:
{job_description}

Resume:
{resume_text}
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a professional ATS evaluator."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
    )

    return response.choices[0].message.content
import streamlit as st

def show_detailed_ats_analysis(analysis_text):
    st.markdown("## 📊 Detailed ATS Analysis")

    # Large scrollable container
    st.markdown("""
        <style>
        .big-ats-box {
            height: 450px;
            overflow-y: scroll;
            padding: 20px;
            border-radius: 12px;
            background-color: #0e1117;
            border: 1px solid #2e2e2e;
            font-size: 16px;
            line-height: 1.6;
            white-space: pre-wrap;
        }
        </style>
    """, unsafe_allow_html=True)

    st.markdown(
        f'<div class="big-ats-box">{analysis_text}</div>',
        unsafe_allow_html=True
    )
    if ats_result:
     show_detailed_ats_analysis(ats_result)

# -------- Generate PDF Report (Saved to Disk) --------
def generate_pdf_file(content):

    file_path = "ATS_Report.pdf"
    doc = SimpleDocTemplate(file_path, pagesize=A4)
    styles = getSampleStyleSheet()
    story = []

    for line in content.split("\n"):
        safe_line = line.replace("&", "&amp;")
        story.append(Paragraph(safe_line, styles["Normal"]))
        story.append(Spacer(1, 6))

    doc.build(story)
    return file_path


# -------- Main Function --------
def analyze_resume(resume_file, job_description):

    try:
        if resume_file is None:
            return "Please upload a resume PDF.", None

        resume_text = extract_text_from_pdf(resume_file)

        if not resume_text.strip():
            return "Could not extract text. Upload a text-based PDF.", None

        result = generate_analysis(resume_text, job_description)

        pdf_path = generate_pdf_file(result)

        return result, pdf_path

    except Exception as e:
        return f"ERROR OCCURRED:\n\n{str(e)}", None


# -------- Gradio UI --------
with gr.Blocks() as demo:

    gr.Markdown("## AI Resume Analyzer - Stable Final Version")

    resume_input = gr.File(
        type="filepath",
        file_types=[".pdf"],
        label="Upload Resume (PDF)"
    )

    job_input = gr.Textbox(
        label="Paste Job Description",
        lines=8
    )

    btn = gr.Button("Analyze Resume")

    output_text = gr.Textbox(label="Detailed ATS Analysis")
    output_file = gr.File(label="Download ATS Report PDF")

    btn.click(
        analyze_resume,
        inputs=[resume_input, job_input],
        outputs=[output_text, output_file]
    )


demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c1dd9f5945be75dd78.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
